<a href="https://colab.research.google.com/github/zbeyza/Advanced-Functional-EDA/blob/main/supervised_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# install required packages
%pip install "transformers>=4.36.0" "trl>=0.7.0" "datasets>=2.14.0" "torch>=2.0.0"
%pip install "accelerate>=0.24.0" "peft>=0.7.0" "trackio"

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
import json
from typing import Optional, Dict, Any

if torch.cuda.is_available():
  device = "cuda"
  print(f"using CUDA GPU {torch.cuda.get_device_name(0)}")
  print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
elif hasattr(torch.backends, "mps") and torch.backend.mps.is_available():
  device = "mps"
  print("Using Apple MPS")
else:
  device = "cpu"
  print("Using CPU - need GPU to train models")

# authenticate with huggingface
from huggingface_hub import login

using CUDA GPU NVIDIA L4
GPU memory: 23.7GB


In [14]:
# loading SmolLM3 models

# base and instruct model for comparison
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

# tokenizers
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# models
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

print("models loaded successfully")

[transformers] Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

models loaded successfully


In [16]:
# chat template formatting
# create different types of conversations to test

conversations = {
    "simple_qa": [
        {"role": "user", "content": "what is machine learning?"}
    ],

    "with_system": [
        {"role": "system", "content": "you are a helpful AI assistantspecialized in explaining technical concepts clearly."},
        {"role": "user", "content": "what is machien learninf?"}
    ],

    "multi_turn": [
        {"role": "system", "content": "you are a math ttor."},
        {"role": "user", "content": "what is calculus?"},
        {"role": "asssitant", "content": "calculus is a branch of mathematics that deals with rates of change and accumulation of quantities."},
        {"role": "user", "content": "can you give me a simple example?"}
    ],

    "reasoning_task": [
        {"role": "user", "content": "solve step by step: if a train travels 120 miles in 2 hours, what is its average speed?"}
    ]
}

for conv_type, messages in conversations.items():
  print(f"--- {conv_type.upper()} ---")

  # format without gneration prompt (for completed conversations)
  formatted_complete = instruct_tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=False
  )

  # format with generation prompt (for inference)
  formatted_prompt = instruct_tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  print("complete conversation format")
  print(formatted_complete)
  print("\nWith genration prompt:")
  print(formatted_prompt)
  print("\n" + "="*60 + "\n")

--- SIMPLE_QA ---
complete conversation format
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 20 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current s

In [ ]:
# base vs instruct model responses